# Задание

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.
Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку. 

# Загрузка и первичное изучение данных

Начнем работу с проверки данных.

In [2]:
import pandas as pd
from nltk.stem import SnowballStemmer 
russian_stemmer = SnowballStemmer('russian') # Для работы с лемами 

data = pd.read_csv(r'C:\Users\alexa\Downloads\КУРС ПИТОН\data.csv')
data = data.drop_duplicates() # Удаление повторов для корректного анализа
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21471 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21471 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21471 non-null  int64  
 3   education         21471 non-null  object 
 4   education_id      21471 non-null  int64  
 5   family_status     21471 non-null  object 
 6   family_status_id  21471 non-null  int64  
 7   gender            21471 non-null  object 
 8   income_type       21471 non-null  object 
 9   debt              21471 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21471 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.1+ MB


## Проверка отсутствия данных 
При вызове метода info() в столбцах days_employed и total_income замечено одинаковое количество пропусков. Надо убедиться какой они имеют характер системный или нет.

In [3]:
data[
    (data['days_employed'].isna()) & (data['total_income'].isna())
].count()

children            2120
days_employed          0
dob_years           2120
education           2120
education_id        2120
family_status       2120
family_status_id    2120
gender              2120
income_type         2120
debt                2120
total_income           0
purpose             2120
dtype: int64

**Вывод**: пропуски имеют системный характер, так как пропущенны сразу две характеристики у определенных клиентов. Для далнейшего анализа потрубуется замена total_income медианами значениями для каждой категории income_type, так как это важная характеристика в анализе. Просто удалить эти данные не получится, потому что это 9,8% всех данных. Ошибка имеет очень большой вес и скорее всего возникла из-за человечского фактора, так что надо обновить данные по данным клиентам, чтобы к следущему анализу были более точные данные.

## Замена переменных

Пока заменим пустые знаения days_employed на 0, так как для анализа они не будут нам нужны, но их тоже нужно уточнить у клиентов. Так же заменим тип данных с float64 на int64 для оптимизации хранения и последующей обработки информации.

In [4]:
data['days_employed'] = data['days_employed'].fillna(0)
data['days_employed'] = data['days_employed'].astype('int64')

Так же заменим значения total_income как предлагалось в выводе выше и изменим тип с float64 на int64 для оптимизации хранения и последующей обработки информации.

In [5]:
median_inc = data.groupby('income_type')['total_income'].median()
for income_type in median_inc.index:
    data.loc[
        (data['income_type'] == income_type) & (data['total_income'].isna()), 
        'total_income'
    ] = median_inc[income_type]
data['total_income'] = data['total_income'].astype('int64')
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21471 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   children          21471 non-null  int64 
 1   days_employed     21471 non-null  int64 
 2   dob_years         21471 non-null  int64 
 3   education         21471 non-null  object
 4   education_id      21471 non-null  int64 
 5   family_status     21471 non-null  object
 6   family_status_id  21471 non-null  int64 
 7   gender            21471 non-null  object
 8   income_type       21471 non-null  object
 9   debt              21471 non-null  int64 
 10  total_income      21471 non-null  int64 
 11  purpose           21471 non-null  object
dtypes: int64(7), object(5)
memory usage: 2.1+ MB


## Проверка возраста 
Посмотрим какие вообще есть значения возраста.

In [6]:
data['dob_years'].value_counts()

dob_years
35    616
40    607
41    606
34    601
38    597
42    596
33    581
39    572
31    559
36    554
44    545
29    544
30    538
48    537
37    536
50    513
43    512
32    509
49    508
28    503
45    497
27    493
52    484
56    484
47    477
54    476
46    473
53    459
57    456
58    456
51    448
55    443
59    443
26    408
60    374
25    357
61    354
62    349
63    269
24    264
64    262
23    253
65    194
22    183
66    182
67    167
21    111
0     101
68     99
69     85
70     65
71     58
20     51
72     33
19     14
73      8
74      6
75      1
Name: count, dtype: int64

Проверим возраст клиентов, чтобы позже коректно сравнивать его с рабочим стажем. При проверке надо посмотреть есть ли клиенты младше 18 лет, так как им кредит выдать нельзя по закону.

In [7]:
data[
    (data['dob_years'] < 18) 
].groupby('dob_years').agg({'dob_years': ['count']})

,dob_years
,count
dob_years,
0,101


Как видно единственное не подходящее условие это 0 лет.

Посмтрим находятся ли пропуски в одной категории income_type или нет.

In [8]:
data['income_type'][
    (data['dob_years'] < 18) 
].value_counts()

income_type
сотрудник      55
пенсионер      20
компаньон      20
госслужащий     6
Name: count, dtype: int64

**Вывод**: Надо уточнить у клиентов повторно их возраст, потому что в далнейшем эти данные могут понадобиться. Пока для анализа факторов, которые влиют на погашения кредита в срок возраст нам не нужен.

## Проверка стаж больше возраста
Стаж не может быть больше возраста, посмотрим есть ли такие значение в days_employed. (Берется модуль стажа, так как некоторые значение в столбце days_employed отрицательные)

In [9]:
data['days_employed'][
    (data['dob_years'] < abs(data['days_employed'])/365) & (data['dob_years'] != 0)
].sort_values()

20444    328728
9328     328734
17782    328771
14783    328795
7229     328827
          ...  
7794     401663
2156     401674
7664     401675
10006    401715
6954     401755
Name: days_employed, Length: 3428, dtype: int64

Делаем мини вывод, что все значение в данном случае положительны и можно пока выдвинуть гипотизу, что положительные числа говорят, а сбое записи рабочего стажа.

Посчитаем сколько всего положительных записей в солбце days_employed в изнначальной данных.

In [10]:
data[data['days_employed'] > 0]['days_employed'].count()

np.int64(3445)

Посчитаем сколько всего положительных записей в солбце days_employed при возрасте 0.

In [11]:
data['days_employed'][
    (data['days_employed'] > 0) & (data['dob_years'] == 0)
].count()

np.int64(17)

Гипотеза подтвердилась: все 3445 записей с положительным стажем работы относятся к клиентам у кого возраст меньше стажа (3428 записей) или возраст равен 0 лет (17 записей).

Посмотрим выборку клиентов, у кого значение days_employed > 0.

In [12]:
data[data['days_employed'] > 0]['income_type'].value_counts()

income_type
пенсионер      3443
безработный       2
Name: count, dtype: int64

Посмотрим сколько всего пенсионеров и бездомных в базе данных.

In [13]:
data['income_type'][
    (data['income_type'].str.lower() == 'пенсионер') | 
    (data['income_type'].str.lower() == 'безработный')
].value_counts()

income_type
пенсионер      3837
безработный       2
Name: count, dtype: int64

Посмотрим сколько всего пенсионеров с пустым значением days_employed.

In [14]:
data['income_type'][
    (data['income_type'].str.lower() == 'пенсионер') &
    (data['days_employed'] == 0)
].value_counts()

income_type
пенсионер    394
Name: count, dtype: int64

**Вывод**: Почему-то у всех пенсионеров и бездомных сбит рабочий стаж days_employed. Он иммет положительное значение, которое выбивается из общей выборки, так как в основном они отрицательное, что говорит о ситемной ошибке, потому что все представители из этих двух групп имеет ошибку в данном столбце. (Пенсинеры у которых отсутствует значение days_employed попали под первую ошибку, которая разобрана выше)

Положительные значения, а так же некоторая часть пустых значений записаных у пенсеионеров можно заменить на среднее значение стажа, так как на пенсию все выходят +- в одно время. Возьмем возраст начала работы равный 20 годам, а возраст выхода на пенсию: 60 лет (женсикй пол) и 65 лет (мужской пол). 

In [15]:
data.loc[
    (data['gender'] == 'F') & (data['income_type'].str.lower() == 'пенсионер'), 
    'days_employed'
    ] = 14600
data.loc[
    (data['gender'] == 'M') & (data['income_type'].str.lower() == 'пенсионер'), 
    'days_employed'
    ] = 16425

## Проверка количества детей
Посмотрим на количество детей, какие вообще есть значения, возможно там тоже есть опечатки.

In [16]:
data['children'].value_counts()

children
 0     14107
 1      4809
 2      2052
 3       330
 20       76
-1        47
 4        41
 5         9
Name: count, dtype: int64

Из данных виден очень странный скачок между 5 и 20 детьми, а так же 20 детей звучит достаточно нереально... Скорее всего при вводе была допущенна опечатка и имелось ввиду 2 ребенка. Так же значение -1 тоже невозможное, скорее всего возникла такая же опечатка и имелся ввиду 1 ребенок.

In [17]:
data.loc[data['children'] == 20, 'children'] = 2 
data.loc[data['children'] == -1, 'children'] = 1

## Проверка значений total_income
Посмотрим какие есть значение total_income

In [18]:
data['total_income'].value_counts().sort_index()

total_income
20667      1
21205      1
21367      1
21695      1
21895      1
          ..
1711309    1
1715018    1
1726276    1
2200852    1
2265604    1
Name: count, Length: 18608, dtype: int64

**Вывод**: Сам доход распределен плавно и нет никаких анамольных скачков. Так что тут все в порядке

## Проверка соответствия айди и категории
Проверим соответствие семейного статуса с айди 

In [19]:
data.groupby('family_status_id')['family_status'].unique()

family_status_id
0          [женат / замужем]
1         [гражданский брак]
2           [вдовец / вдова]
3                [в разводе]
4    [Не женат / не замужем]
Name: family_status, dtype: object

К одному айдишнику не привязано несколько статусов. Все хорошо)

Проверим соответствие учебной категории с айди

In [20]:
data.groupby('education_id')['education'].unique()

education_id
0                             [высшее, ВЫСШЕЕ, Высшее]
1                          [среднее, Среднее, СРЕДНЕЕ]
2    [неоконченное высшее, НЕОКОНЧЕННОЕ ВЫСШЕЕ, Нео...
3                    [начальное, НАЧАЛЬНОЕ, Начальное]
4     [Ученая степень, УЧЕНАЯ СТЕПЕНЬ, ученая степень]
Name: education, dtype: object

Для удобства заменим все значения на строчный формат

In [21]:
data['education'] = data['education'].str.lower()

## Проверка значений income_type
Проанализируем столбец тип дохода

In [22]:
data['income_type'].value_counts()

income_type
сотрудник          11091
компаньон           5080
пенсионер           3837
госслужащий         1457
безработный            2
предприниматель        2
студент                1
в декрете              1
Name: count, dtype: int64

Студент, компаньон и в декрете - это не тип дохода. Эти статусы отвечают за другие характиристики, поэтому студент становится - безработным, так же как и компаньон, потому что доход партнера не является доходом клиента. Женщина в декрете становится - сотрудником.

In [23]:
data.loc[data['income_type'] == 'компаньон','income_type'] = 'безработный'
data.loc[data['income_type'] == 'студент','income_type'] = 'безработный'
data.loc[data['income_type'] == 'в декрете','income_type'] = 'сотрудник'

## Проверка категорий кредитов (лематизация)
Рассмотрим категории на которые люди берут кредит

In [24]:
data['purpose'].value_counts()

purpose
свадьба                                   793
на проведение свадьбы                     773
сыграть свадьбу                           769
операции с недвижимостью                  675
покупка коммерческой недвижимости         662
операции с жильем                         652
покупка жилья для сдачи                   652
операции с коммерческой недвижимостью     650
жилье                                     646
покупка жилья                             646
покупка жилья для семьи                   638
строительство собственной недвижимости    635
недвижимость                              633
операции со своей недвижимостью           627
строительство жилой недвижимости          625
покупка недвижимости                      621
покупка своего жилья                      620
строительство недвижимости                619
ремонт жилью                              607
покупка жилой недвижимости                606
на покупку своего автомобиля              505
заняться высшим образовани

Сделаем лематизацию, чтобы провести более точный анализ. Создадим функцию, которая соеденит разные категории по общим словам.

In [25]:
def complex_categorization(text):
    text = text.lower()
    words = text.split()
    stemmed_words = [russian_stemmer.stem(w) for w in words]
    stemmed_text = ' '.join(stemmed_words)
    
    if 'свадьб' in stemmed_words:
        return 'свадьба'
    elif 'автомоб' in stemmed_words or 'автомобил' in stemmed_words:
        return 'автомобиль'
    elif 'образован' in stemmed_words:
        return 'образование'
    elif 'ремонт' in stemmed_words:
        return 'ремонт'
    elif 'операц' in stemmed_text:
        if 'недвижим' in stemmed_text or 'жил' in stemmed_text:
            return 'операции с недвижимостью'
    elif 'строительств' in stemmed_text:
        if 'недвижим' in stemmed_text or 'жил' in stemmed_text:
            return 'строительство недвижимости'
    elif 'недвижим' in stemmed_text or 'жил' in stemmed_text:
        return 'покупка недвижимости'
        
data['purpose'] = data['purpose'].apply(complex_categorization)

In [26]:
data['purpose'].value_counts()

purpose
покупка недвижимости          5724
автомобиль                    4308
образование                   4014
операции с недвижимостью      2604
свадьба                       2335
строительство недвижимости    1879
ремонт                         607
Name: count, dtype: int64

# Перейдем к аналитике

## Влияние количества детей на погашения кредита
Проверим как влияет количество детей на возврат кредита

In [27]:
mean_table_ch = data.pivot_table(
    index='children', columns='income_type', values='debt', aggfunc='mean'
).round(3)
count_table_ch = data.pivot_table(
    index='children', columns='income_type', values='debt', aggfunc='count'
)
combined_ch = mean_table_ch.astype(str) + ' (' + count_table_ch.astype(str) + ')'
combined_ch

income_type,безработный,госслужащий,пенсионер,предприниматель,сотрудник
children,,,,,
0,0.072 (3141.0),0.068 (866.0),0.056 (3518.0),0.0 (2.0),0.088 (6580.0)
1,0.082 (1308.0),0.053 (358.0),0.049 (283.0),nan (nan),0.105 (2907.0)
2,0.071 (551.0),0.032 (189.0),0.103 (29.0),nan (nan),0.113 (1359.0)
3,0.063 (79.0),0.056 (36.0),0.167 (6.0),nan (nan),0.091 (209.0)
4,0.0 (2.0),0.0 (7.0),0.0 (1.0),nan (nan),0.129 (31.0)
5,0.0 (2.0),0.0 (1.0),nan (nan),nan (nan),0.0 (6.0)


Разделим данные на 2 категории: *Уверенная выборка* (количество больше 500) и *Неуверенная выборка* (количество меньше 500)

*Уверенная выборка* 

Высокий риск (>10%):

- Сотрудники с 2 детьми - 11,3% (n=1359)
- Сотрудники с 1 ребенком - 10,5% (n=2907)
  
Средний риск (6-10%):

- Сотрудники без детей - 8,8% (n=6580)
- Безработные с 1 ребенком - 8,2% (n=1308)
- Безработные без детей - 7,2% (n=3141)
- Безработные с 2 детьми - 7,1% (n=551)
- Госслужащие без детей - 6,8% (n=866)
  
Низкий риск (<6%):

- Пенсионеры без детей - 5,6% (n=3518)

Из *Уверенной выборки* видно, что сотрудники с детьми - группа повышенного риска, где вероятность невозврата превышает 10%, а сотрудники без детей так же имеют высокие показатели 8,8% . Безработные демонстрируют стабильно средний риск на уровне 7-8%, независимо от наличия детей. Среди всех крупных сегментов пенсионеры без детей имеют самый низкий риск - около 5,6%. Госслужащие без детей также относятся к средней категории риска.

*Неуверенная выборка*

Высокий риск (>10%):

- Сотрудники с 4 детьми - 12,9% (n=31)
- Пенсионеры с 3 детьми - 16,7% (n=6)
- Пенсионеры с 2 детьми - 10,3% (n=29)
  
Средний риск (6-10%):

- Сотрудники с 3 детьми - 9,1% (n=209)
- Госслужащие с 1 ребенком - 5,3% (n=358)
- Безработные с 3 детьми - 6,3% (n=79)
- Госслужащие с 3 детьми - 5,6% (n=36)
  
Низкий риск (<6%):

- Пенсионеры с 1 ребенком - 4,9% (n=283)
- Госслужащие с 2 детьми - 3,2% (n=189)
- Предприниматель 

Для всех групп из *Неуверенной выборки* особое внимание и индивидуальный подход, так как данных недостаточно. Критическое внимание к группе восокого риска, предпринимателям, а так же клиентам с 4-5 детьми, так как некоторых данных вообще нет.

## Влияние семейного положения на погашения кредита
Проверим как влияет семейное положение на возврат кредита

In [28]:
mean_table_fm = data.pivot_table(
    index='family_status', columns='income_type', values='debt', aggfunc='mean'
).round(3)
count_table_fm = data.pivot_table(
    index='family_status', columns='income_type', values='debt', aggfunc='count'
)
combined_fm = mean_table_fm.astype(str) + ' (' + count_table_fm.astype(str) + ')'
combined_fm

income_type,безработный,госслужащий,пенсионер,предприниматель,сотрудник
family_status,,,,,
Не женат / не замужем,0.089 (807.0),0.073 (164.0),0.046 (351.0),nan (nan),0.117 (1488.0)
в разводе,0.029 (272.0),0.049 (81.0),0.059 (219.0),nan (nan),0.096 (623.0)
вдовец / вдова,0.093 (97.0),0.044 (45.0),0.069 (537.0),nan (nan),0.054 (280.0)
гражданский брак,0.094 (1014.0),0.069 (262.0),0.055 (655.0),0.0 (1.0),0.107 (2231.0)
женат / замужем,0.067 (2893.0),0.055 (905.0),0.055 (2075.0),0.0 (1.0),0.089 (6470.0)


Анализ влияния семейного статуса на риск невозврата кредита. Деление на группы остается как выше для всех последующих примеров. Для удобства это больше не будет упоменаться. 

*Уверенная выборка* 

Высокий риск (>10%):

- Сотрудники, не женаты/не замужем - 11,7% (n=1488)
- Сотрудники в гражданском браке - 10,7% (n=2231)
  
Средний риск (6-10%):

- Безработные, не женаты/не замужем - 8,9% (n=807)
- Сотрудники в разводе - 9,6% (n=623)
- Пенсионеры, вдовцы/вдовы - 6,9% (n=537)
- Безработные в гражданском браке - 9,4% (n=1014)
- Безработные, женаты/замужем - 6,7% (n=2893)
- Сотрудники, женаты/замужем - 8,9% (n=6470)

Низкий риск (<6%):

- Пенсионеры в гражданском браке - 5,5% (n=655)
- Госслужащие, женаты/замужем - 5,5% (n=905)
- Пенсионеры, женаты/замужем - 5,5% (n=2075)

Среди *Уверенной выборки* наибольший риск демонстрируют сотрудники без официального брака - как не состоящие в браке 11,7%, так и состоящие в гражданском браке 10,7%. Безработные показывают средний уровень риска вне зависимости от семейного статуса. Наиболее надежными оказываются пенсионеры и госслужащие в официальном браке с риском около 5,5%.

*Неуверенная выборка* 

Средний риск (6-10%):

- Госслужащие, не женаты/не замужем - 7,3% (n=164)
- Безработные, вдовцы/вдовы - 9,3% (n=97)
- Госслужащие в гражданском браке - 6,9% (n=262)

Низкий риск (<6%):

- Пенсионеры, не женаты/не замужем - 4,6% (n=351)
- Безработные в разводе - 2,9% (n=272)
- Пенсионеры в разводе - 5,9% (n=219)
- Госслужащие в разводе - 4,9% (n=81)
- Госслужащие, вдовцы/вдовы - 4,4% (n=45)
- Сотрудники, вдовцы/вдовы - 5,4% (n=280)
- Предприниматели (все статусы) - 0,0% (n=2)

В *Неуверенной выборке* отсутствуют группы с высоким риском. Особого внимания требуют предприниматели - несмотря на нулевой риск, данные крайне ограничены (всего 2 наблюдения). Все группы из *Неуверенной выборки* требуют индивидуального подхода из-за недостаточности данных.

## Влияние уровнем дохода на погашения кредита

### Обозначим уровни дохода как:
- Очень низкий - меньше 0.5 медианного значения
- Низкий - от 0.5 медианного значения до 0.75 медианного значения
- Средний - от 0.75 медианного значения до 1.25 медианного значения
- Выше среднего - от 1.25 медианного значения до 2 медианного значения
- Высокий - от 2 медианного значения до 3 медианного значения
- Очень высокий - болше 3 медианного значения


In [29]:
med = data['total_income'].median()
def level_income(total_income):
    if total_income <= med * 0.5:
        return 'очень низкий'
    elif total_income > med * 0.5 and total_income <= med * 0.75:
        return 'низкий'
    elif total_income > med * 0.75 and total_income <= med * 1.25:
        return 'средний'
    elif total_income > med * 1.25 and total_income <= med * 2:
        return 'выше среднего'
    elif total_income > med * 2 and total_income <= med * 3:
        return 'высокий'
    else:
        return 'очень высокий'
data['level_income'] = data['total_income'].apply(level_income)
data['level_income'].value_counts()

level_income
средний          9628
выше среднего    4790
низкий           3717
очень низкий     1579
высокий          1348
очень высокий     409
Name: count, dtype: int64

Проверим как влияет уровень дохода на возврат кредита. 

In [30]:
mean_table_inc = data.pivot_table(
    index='level_income', columns='income_type', values='debt', aggfunc='mean'
).round(3)
count_table_inc = data.pivot_table(
    index='level_income', columns='income_type', values='debt', aggfunc='count'
)
combined_inc = mean_table_inc.astype(str) + ' (' + count_table_inc.astype(str) + ')'
combined_inc

income_type,безработный,госслужащий,пенсионер,предприниматель,сотрудник
level_income,,,,,
высокий,0.069 (535.0),0.019 (103.0),0.04 (125.0),nan (nan),0.096 (585.0)
выше среднего,0.069 (1452.0),0.072 (335.0),0.07 (582.0),nan (nan),0.082 (2421.0)
низкий,0.083 (588.0),0.073 (248.0),0.045 (862.0),nan (nan),0.102 (2019.0)
очень высокий,0.056 (195.0),0.0 (31.0),0.028 (36.0),0.0 (2.0),0.083 (145.0)
очень низкий,0.082 (147.0),0.061 (99.0),0.052 (578.0),nan (nan),0.081 (755.0)
средний,0.078 (2166.0),0.056 (641.0),0.06 (1654.0),nan (nan),0.102 (5167.0)


Анализ влияния уровня дохода и типа занятости на риск невозврата кредита

*Уверенная выборка* 

Высокий риск (>10%):

- Сотрудники с низким доходом - 10,2% (n=2019)
- Сотрудники со средним доходом - 10,2% (n=5167)

Средний риск (6-10%):

- Безработные с высоким доходом - 6,9% (n=535)
- Сотрудники с высоким доходом - 9,6% (n=585)
- Безработные с доходом выше среднего - 6,9% (n=1452)
- Пенсионеры с доходом выше среднего - 7,0% (n=582)
- Сотрудники с доходом выше среднего - 8,2% (n=2421)
- Безработные с низким доходом - 8,3% (n=588)
- Безработные со средним доходом - 7,8% (n=2166)
- Сотрудники с очень низким доходом - 8,1% (n=755)
- Пенсионеры со средним доходом - 6,0% (n=1654)

Низкий риск (<6%):

- Пенсионеры с низким доходом - 4,5% (n=862)
- Пенсионеры с очень низким доходом - 5,2% (n=578)
- Госслужащие со средним доходом - 5,6% (n=641)

Из *Уверенной выборки* видно, что сотрудники с низким и средним доходом демонстрируют повышенный риск невозврата более 10%, но для анализа стоит учитывать, что у 9,8% пользователей отсутствовал доход и он был заменен медианам значениями, так что в данных есть искожания. Сотрудники с другими уровнями дохода, а также безработные и пенсионеры (кроме пенсионеров с низким и очень низким доходом) показывают средний риск. Наиболее надежными оказываются пенсионеры с низким доходом и госслужащие со средним доходом.

*Неуверенная выборка*

Средний риск (6-10%):

- Госслужащие с доходом выше среднего - 7,2% (n=335)
- Госслужащие с низким доходом - 7,3% (n=248)
- Сотрудники с очень высоким доходом - 8,3% (n=145)
- Безработные с очень низким доходом - 8,2% (n=147)
- Госслужащие с очень низким доходом - 6,1% (n=99)

Низкий риск (<6%):

- Госслужащие с высоким доходом - 1,9% (n=103)
- Пенсионеры с высоким доходом - 4,0% (n=125)
- Безработные с очень высоким доходом - 5,6% (n=195)
- Госслужащие с очень высоким доходом - 0,0% (n=31)
- Пенсионеры с очень высоким доходом - 2,8% (n=36)
- Предприниматели с очень высоким доходом - 0,0% (n=2)

Среди *Неуверенной выборки* отсутствуют группы с высоким риском. Особого внимания требуют предприниматели - несмотря на нулевой риск, данные крайне ограничены (всего 2 наблюдения). Так же пристальное вниманеи к сотрудником с очень высоким доходом 8,3% и к безработным с очень низким доходом 8,2%. Все группы из *Неуверенной выборки* требуют индивидуального подхода из-за недостаточности данных.

## Влияние разных целей кредита на его погашения 


In [31]:
mean_table_pur = data.pivot_table(
    index='purpose', columns='income_type', values='debt', aggfunc='mean'
).round(3)
count_table_pur = data.pivot_table(
    index='purpose', columns='income_type', values='debt', aggfunc='count'
)
combined_pur = mean_table_pur.astype(str) + ' (' + count_table_pur.astype(str) + ')'
combined_pur

income_type,безработный,госслужащий,пенсионер,предприниматель,сотрудник
purpose,,,,,
автомобиль,0.081 (1052.0),0.077 (286.0),0.064 (796.0),nan (nan),0.113 (2174.0)
образование,0.075 (954.0),0.081 (258.0),0.066 (722.0),nan (nan),0.11 (2080.0)
операции с недвижимостью,0.068 (620.0),0.058 (172.0),0.045 (467.0),nan (nan),0.098 (1345.0)
покупка недвижимости,0.067 (1321.0),0.043 (422.0),0.051 (984.0),0.0 (1.0),0.08 (2996.0)
ремонт,0.048 (146.0),0.073 (41.0),0.049 (102.0),nan (nan),0.063 (318.0)
свадьба,0.099 (527.0),0.044 (159.0),0.058 (428.0),0.0 (1.0),0.084 (1220.0)
строительство недвижимости,0.065 (463.0),0.042 (119.0),0.047 (338.0),nan (nan),0.097 (959.0)


Анализ влияния цели кредита и типа занятости на риск невозврата

*Уверенная выборка*

Высокий риск (>10%):

- Сотрудники, берущие кредит на автомобиль - 11,3% (n=2174)
- Сотрудники, берущие кредит на автомобиль - 11,3% (n=2174)
- Сотрудники, берущие кредит на образование - 11,0% (n=2080)

Средний риск (6–10%):
  
- Безработные, берущие кредит на свадьбу - 9,9% (n=527)
- Сотрудники на операции с недвижимостью - 9,8% (n=1345)
- Сотрудники на строительство недвижимости - 9,7% (n=959)
- Безработные на автомобиль - 8,1% (n=1052)
- Безработные на образование - 7,5% (n=954)
- Пенсионеры на автомобиль - 6,4% (n=796)
- Пенсионеры на образование - 6,6% (n=722)
- Сотрудники на покупку недвижимости - 8,0% (n=2996)
- Безработные на покупку недвижимости - 6,7% (n=1321)
- Безработные на операции с недвижимостью - 6,8% (n=620)

Низкий риск (<6%):

- Пенсионеры на покупку недвижимости - 5,1% (n=984)
- Пенсионеры на операции с недвижимостью - 4,5% (n=467)

Из *Уверенной выборки* видно, что сотрудники, берущие кредит на автомобиль или образование, демонстрируют повышенный риск невозврата более 11%. Безработные, оформляющие кредит на свадьбу, также показывают высокий риск 9,9% вместе с сотрудниками берущими кредит на операции с недвижимостью 9,8% и на строительство недвижимости 9,7%. Наиболее надежными целями кредита среди крупных сегментов являются операции с недвижимостью для пенсионеров 4,5% и покупка недвижомости пенсионерами 5,1%. Интересно, что покупка недвижимости в целом связана с более низким риском по сравнению с другими целями, а так же пенсионеры возращают кредит лучше остальных групп.

Неуверенная выборка

Средний риск (6–10%):

- Сотрудники на ремонт - 6,3% (n=318)
- Госслужащие на автомобиль - 7,7% (n=286)
- Госслужащие на образование - 8,1% (n=258)
- Госслужащие на ремонт - 7,3% (n=41)

Низкий риск (<6%):

- Госслужащие на покупку недвижимости - 4,3% (n=422)
- Безработные на ремонт - 4,8% (n=146)
- Пенсионеры на ремонт - 4,9% (n=102)
- Госслужащие на операции с недвижимостью - 5,8% (n=172)
- Госслужащие на строительство недвижимости - 4,2% (n=119)
- Пенсионеры на свадьбу - 5,8% (n=428)
- Предприниматели на покупку недвижимости - 0,0% (n=1)
- Предприниматели на свадьбу - 0,0% (n=1)

Среди *Неуверенной выборки* отсутствуют группы с высоким риском. Особого внимания требуют предприниматели - несмотря на нулевой риск, данные крайне ограничены (всего 2 наблюдения), а так же госслужащие берущие кредит на автомобиль, образоване и ремонт. Все группы из *Неуверенной выборки* требуют индивидуального подхода из-за недостаточности данных.

# Выводы